In [0]:
# ── 1. 원본 데이터 읽기 ──────────────────────────────────────────

df_ott_contents = spark.table('edu260323.hjh.bronze_ott_contents')
df_ott_logs = spark.table('edu260323.hjh.bronze_ott_logs')
df_ott_users = spark.table('edu260323.hjh.bronze_ott_users')

# ── 1.1 컨텐츠 데이터 확인 ──────────────────────────────────────────

# display(df_ott_contents.limit(3))
# display(df_ott_logs.limit(3))
# display(df_ott_users.limit(3))

In [0]:
from pyspark.sql.functions import col,trim
from delta.tables import DeltaTable

# 2. OTT 컨텐츠 데이터 ──────────────────────────────────────────


# ── 2.1 중복 처리 ──────────────────────────────────────────

# ── 2.1.1 중복데이터 df 생성 ──────────────────────────────────────────
dup1_con = df_ott_contents.dropDuplicates()

dup2_con = (
    df_ott_contents
    .groupBy("content_id") 
    .count()
    .filter(col("count") > 1)
)

# ── 2.1.2 중복행 df join ──────────────────────────────────────────
dup_rows = (
    df_ott_contents
    .join(dup2_con, on="content_id", how="inner")
)

# ── 2.1.3 정상행 df join ──────────────────────────────────────────
normal_rows = (
    df_ott_contents
    .join(dup2_con, on="content_id", how="leftanti")
)

# ── 2.2 공백제거 ──────────────────────────────────────────
trim_normal_rows = (
    normal_rows
    .withColumn("content_id", trim(col("content_id")))
    .withColumn("Content_type", trim(col("Content_type")))
    .withColumn("Genre", trim(col("Genre")))
)

# ── 2.3 저장 ──────────────────────────────────────────
# ── 2.3.1 중복테이블 저장 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_contents_dup"):
    (
    dup_rows.write
    .saveAsTable("edu260323.hjh.silver_ott_contents_dup")
    )

else:(
    dup_rows.write
    .mode("append")
    .saveAsTable("edu260323.hjh.silver_ott_contents_dup")
    )

# ── 2.3.2 정상테이블 저장 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_contents"):
    (
    trim_normal_rows.write
    .saveAsTable("edu260323.hjh.silver_ott_contents")
    )

else:
    target_con = DeltaTable.forName(spark, "edu260323.hjh.silver_ott_contents")
    (
        target_con.alias("target")
        .merge(
            trim_normal_rows.alias("source"),
            "target.content_id = source.content_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
from pyspark.sql.functions import coalesce, lit, current_date, col, trim


# 3. ott_logs 데이터 ────────────────────────────────────────────

# ── 3.1 가공 ──────────────────────────────────────────
# ── 3.1.1 중복 df 생성 ──────────────────────────────────────────

dup1_logs = df_ott_logs.dropDuplicates()

dup2_logs = (
    df_ott_logs
    .groupBy("log_id")
    .count()
    .filter(col("count") > 1)
)
# ── 3.1.2 중복 df join ──────────────────────────────────────────
dup_rows_log = (
    df_ott_logs
    .join(dup2_logs, on="log_id", how='inner')
)


# ── 3.1.3 정상 df join ──────────────────────────────────────────
normal_rows_log = (
    df_ott_logs
    .join(dup2_logs, on="log_id", how='leftanti')
)

# ── 3.2 Null 값 처리 ──────────────────────────────────────────
cast_normal_rows_log = (
    normal_rows_log
    .withColumn("user_id", coalesce(col("user_id"), trim(lit("unknown"))))
    .withColumn("log_id", trim(col("log_id")))
    .withColumn("content_id", trim(col("content_id")))
    .withColumn("view_timestamp", trim(col("view_timestamp")))
    .withColumn("watched_seconds", trim(col("watched_seconds")))
    .withColumn("device", trim(col("device")))
    .withColumn("date", current_date())
    
)

# ── 3.3 적재 ──────────────────────────────────────────
# ── 3.3.1 중복 데이터 적재 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_logs_dup"):
    (
    dup_rows_log.write
    .saveAsTable("edu260323.hjh.silver_ott_logs_dup")
    )

else:(
    dup_rows_log.write
    .mode("append")
    .saveAsTable("edu260323.hjh.silver_ott_logs_dup")
)
# ── 3.3.2 정상 데이터 적재 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_logs"):
    (
    cast_normal_rows_log.write
    .saveAsTable("edu260323.hjh.silver_ott_logs")
    )

else:
    target_log = DeltaTable.forName(spark, "edu260323.hjh.silver_ott_logs")
    (
        target_log.alias("target")
        .merge(
            cast_normal_rows_log.alias("source"),
            "target.log_id = source.log_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )    

In [0]:
display(df_ott_users.limit(2))
from pyspark.sql.functions import coalesce, lit, current_date, col


# 4. ott_users 데이터 ────────────────────────────────────────────

# ── 4.1 가공 ──────────────────────────────────────────
# ── 4.1.1 중복 df 생성 ──────────────────────────────────────────

dup1_users = df_ott_users.dropDuplicates()

dup2_users = (
    df_ott_users
    .groupBy("user_id")
    .count()
    .filter(col("count") > 1)
)
# ── 4.1.2 중복 df join ──────────────────────────────────────────
dup_rows_users = (
    df_ott_users
    .join(dup2_users, on="user_id", how='inner')
)


# ── 4.1.3 정상 df join ──────────────────────────────────────────
normal_rows_users = (
    df_ott_users
    .join(dup2_users, on="user_id", how='leftanti')
)

# ── 4.2 가공(공백제거) ──────────────────────────────────────────
cast_normal_rows_users = (
    normal_rows_users
    .withColumn("user_id", coalesce(col("user_id"), trim(col("user_id"))))
    .withColumn("plan_type", coalesce(col("plan_type"), trim(col("plan_type"))))
    .withColumn("country", coalesce(col("country"), trim(col("country"))))
    .withColumn("signup_date", coalesce(col("signup_date"), trim(col("signup_date"))))
    .withColumn("date", current_date())
    
)

# ── 4.3 적재 ──────────────────────────────────────────
# ── 4.3.1 중복 데이터 적재 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_users_dup"):
    (
    dup_rows_users.write
    .saveAsTable("edu260323.hjh.silver_ott_users_dup")
    )

else:
    (
    dup_rows_users.write
    .mode("append")
    .saveAsTable("edu260323.hjh.silver_ott_users_dup")
    )

# ── 4.3.2 정상 데이터 적재 ──────────────────────────────────────────
if not spark.catalog.tableExists("edu260323.hjh.silver_ott_users"):
    (
    cast_normal_rows_users.write
    .saveAsTable("edu260323.hjh.silver_ott_users")
    )

else:
    target_users = DeltaTable.forName(spark, "edu260323.hjh.silver_ott_users")
    (
        target_users.alias("target" )
        .merge(
            cast_normal_rows_users.alias("source"),
            "target.user_id = source.user_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

user_id,plan_type,country,signup_date
U00001,Standard,BR,2023-03-05
U00002,Standard,FR,2024-04-25
